<a href="https://colab.research.google.com/github/Duongnb58/BFT65B_11230945_Final/blob/main/notebooks/01_acquisition_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 1. Acquisition and Storage
## Notebook A — FDIC acquisition and integrity verification

**Research scope.** Financials: 1984Q1–2026Q2 (170 reporting quarters). Failures and Institutions: complete source tables. The three datasets are stored as explicitly typed Snappy Parquet.

**Workflow.** 1–2 establish the execution environment; 3 defines persistent storage; 4 collects the source data; 5 verifies coverage and physical integrity; 6 records the measured input for Notebook B. This notebook does not construct failure labels or perform data cleaning.


### 1. Source code and execution environment

The implementation is versioned in GitHub. Notebook execution occurs in Colab; source records and verification evidence are written to Drive rather than to the repository.


In [3]:
from pathlib import Path
import subprocess
import sys

REPOSITORY = "https://github.com/Duongnb58/BFT65B_11230945_Final.git"
REPO_DIR = Path("/content/BFT65B_11230945_Final")

if REPO_DIR.is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPOSITORY, str(REPO_DIR)], check=True)

assert (REPO_DIR / "run_acquisition.py").is_file()
assert (REPO_DIR / "requirements.txt").is_file()
print("Code directory:", REPO_DIR)


Code directory: /content/BFT65B_11230945_Final


### 2. Dependencies and acquisition tests

Pinned package versions establish the parsing and Parquet environment. The acquisition tests cover API paging, error handling, field conversion, checkpoint integrity and real Parquet round trips using synthetic records. No FDIC API requests are made by the tests.


In [4]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(REPO_DIR / "requirements.txt")],
    check=True,
)

TEST_XML = Path("/content/part1_acquisition_tests.xml")
subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "-rs",
     "--junitxml", str(TEST_XML),
     str(REPO_DIR / "tests" / "test_acquisition.py"),
     str(REPO_DIR / "tests" / "test_parquet_integration.py")],
    cwd=REPO_DIR, check=True,
)

import xml.etree.ElementTree as ET
suites = ET.parse(TEST_XML).getroot().findall(".//testsuite")
assert suites and all(int(s.get("failures", 0)) == 0 and
                      int(s.get("errors", 0)) == 0 and
                      int(s.get("skipped", 0)) == 0 for s in suites),     "Acquisition tests must pass without skipped integration checks"

import pyarrow as pa
print("Acquisition tests: passed, 0 skipped")
print("Python:", sys.version.split()[0], "| PyArrow:", pa.__version__)


Acquisition tests: passed, 0 skipped
Python: 3.13.18 | PyArrow: 21.0.0


### 3. Persistent dataset directory

A dedicated Drive directory contains the acquired partitions, manifest, verification report and UTC logs. Notebook B reads this directory as its sole acquisition input.


In [5]:
from google.colab import drive
drive.mount("/content/drive")

sys.path.insert(0, str(REPO_DIR / "src"))
from fdic_pipeline.workflow import StudyPaths

DRIVE_ROOT = Path("/content/drive/MyDrive/BFT65B_11230945_Final")
PATHS = StudyPaths(DRIVE_ROOT)
DATASET_DIR = PATHS.acquisition
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

print("Study root:", DRIVE_ROOT)
print("Acquisition output:", DATASET_DIR)
print("Financials period: 1984Q1–2026Q2; Failures/Institutions: full tables")


Mounted at /content/drive
Study root: /content/drive/MyDrive/BFT65B_11230945_Final
Acquisition output: /content/drive/MyDrive/BFT65B_11230945_Final/fdic_1984Q1_2026Q2
Financials period: 1984Q1–2026Q2; Failures/Institutions: full tables


### 4. FDIC data acquisition

Each Financials quarter is retrieved by reporting date and checked against the API-reported total. Valid units are committed to the manifest with SHA-256 digests. A rerun after interruption resumes only missing or invalid units; the first execution in this new Drive directory downloads all units. Reference tables are fetched in full. The command finishes with an independent verification pass.


In [6]:
ACQUISITION_CMD = [
    sys.executable, str(REPO_DIR / "run_acquisition.py"),
    "--start", "1984-03-31", "--end", "2026-06-30",
    "--out", str(DATASET_DIR),
    "--page-size", "10000", "--pause", "0.25",
]
subprocess.run(ACQUISITION_CMD, cwd=REPO_DIR, check=True)
print("Acquisition and post-download verification completed.")


Acquisition and post-download verification completed.


### 5. Data completeness and integrity

The final verification report is interpreted together with the acquisition manifest. Quarter coverage, API counts, schema selection, retained failure records without CERT and source checksums are checked before the dataset is released for storage experiments.


In [7]:
from fdic_pipeline.workflow import acquisition_contract

ACQUISITION = acquisition_contract(DATASET_DIR)
assert ACQUISITION["quarters"] == 170
assert ACQUISITION["financial_fields"] == 33

print("Verified quarters:", ACQUISITION["quarters"])
print("Financials observations:", f"{ACQUISITION['financial_rows']:,}")
print("Distinct bank certificates:", f"{ACQUISITION['distinct_financial_certs']:,}")
print("Failure events:", f"{ACQUISITION['failures_rows']:,}")
print("Failures without CERT:", f"{ACQUISITION['failures_missing_cert_rows']:,}")
print("Institutions:", f"{ACQUISITION['institutions_rows']:,}")
print("Manifest SHA-256:", ACQUISITION["manifest_sha256"])
print("Verification SHA-256:", ACQUISITION["verification_report_sha256"])


Verified quarters: 170
Financials observations: 1,682,616
Distinct bank certificates: 24,064
Failure events: 4,117
Failures without CERT: 488
Institutions: 27,835
Manifest SHA-256: 428a95b9cf0cb1b36b3d5571a9488cc57a116b2d28ff887dab1c919a047ec3f4
Verification SHA-256: 3943e51452a1dd7f1e1d19e0fcdfda225fb4a1818d33457b3470476c78154580


### 6. Acquisition evidence and storage handoff

The acquisition stage is complete only when both the manifest and the verification report remain available on Drive. Notebook B reads those exact files, checks their scope and hashes, and constructs an equivalent CSV/Parquet comparison from the verified Financials partitions. The observed counts printed above are the values to use in the report; no count from an earlier study is assumed.


In [8]:
import json

from IPython.display import display, Markdown

with (DATASET_DIR / "manifest.json").open(encoding="utf-8") as f:
    final_manifest = json.load(f)
print("Manifest created (UTC):", final_manifest.get("created_at_utc"))
print("Manifest last updated (UTC):", final_manifest.get("updated_at_utc"))

handoff = (
    f"**Acquisition passed:** {ACQUISITION['quarters']} quarters; "
    f"{ACQUISITION['financial_rows']:,} Financials rows; "
    f"{ACQUISITION['failures_rows']:,} failures; "
    f"{ACQUISITION['institutions_rows']:,} institutions."
)
display(Markdown(handoff))
print("Input directory for Notebook B:", DATASET_DIR)
print("Required evidence:", DATASET_DIR / "manifest.json")
print("Required evidence:", DATASET_DIR / "verification_report.json")


Manifest created (UTC): 2026-11-14T16:06:23+00:00
Manifest last updated (UTC): 2026-11-14T16:21:09+00:00


**Acquisition passed:** 170 quarters; 1,682,616 Financials rows; 4,117 failures; 27,835 institutions.

Input directory for Notebook B: /content/drive/MyDrive/BFT65B_11230945_Final/fdic_1984Q1_2026Q2
Required evidence: /content/drive/MyDrive/BFT65B_11230945_Final/fdic_1984Q1_2026Q2/manifest.json
Required evidence: /content/drive/MyDrive/BFT65B_11230945_Final/fdic_1984Q1_2026Q2/verification_report.json
